# LeetCode 762 — Prime Number of Set Bits in Binary Representation bằng Python

**Link đề:** https://leetcode.com/problems/prime-number-of-set-bits-in-binary-representation/
**Độ khó:** Easy · **Chủ đề:** Math, Bit Manipulation (thao tác bit), Primality Test (kiểm tra số nguyên tố)

```python
class Solution:
    def countPrimeSetBits(self, left: int, right: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **set bit** (bit bằng 1) là gì và đếm nó bằng `bin(x).count("1")`, `x & (x - 1)`, `int.bit_count()`.
2. Ôn lại cách kiểm tra **số nguyên tố** (prime) bằng chia thử đến căn bậc hai.
3. Nhận ra **nút thắt**: số bit 1 của mọi số trong đề đều rất nhỏ (≤ 20) → chỉ cần một tập số nguyên tố cố định.
4. Viết 4 lời giải: trực giác → tập nguyên tố → bitmask "665772" → quy hoạch động (DP) đếm bit.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space và tự giải biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — đếm các số có "số bit 1" là số nguyên tố

Cho hai số nguyên `left` và `right`. Hãy đếm có bao nhiêu số `x` trong đoạn **đóng** `[left, right]` (tính cả hai đầu) mà **số bit 1** trong biểu diễn nhị phân của `x` là một **số nguyên tố**.

Ví dụ `21 = 10101` (nhị phân) có 3 bit 1, và 3 là số nguyên tố.

**Ví dụ 1:** `left = 6, right = 10` → `4`

| x | Nhị phân | Số bit 1 | Nguyên tố? |
|---:|---|:---:|:---:|
| 6 | `110` | 2 | ✔ |
| 7 | `111` | 3 | ✔ |
| 8 | `1000` | 1 | ✘ (1 không phải số nguyên tố) |
| 9 | `1001` | 2 | ✔ |
| 10 | `1010` | 2 | ✔ |

**Ví dụ 2:** `left = 10, right = 15` → `5` (10, 11, 12, 13, 14 đạt; 15 = `1111` có 4 bit, 4 không nguyên tố).

**Constraints:**
- `1 <= left <= right <= 10^6`
- `0 <= right - left <= 10^4`

Câu hỏi trước khi code:
- Làm sao "nhìn" thấy bit của một số trong Python?
- Kiểm tra nguyên tố cho **số bit** (chứ không phải cho `x`) — số bit lớn nhất có thể là bao nhiêu?

In [1]:
# Kiểm tra nhanh đề bài bằng tay với ví dụ 1
for x in range(6, 11):          # range(6, 11) cho 6, 7, 8, 9, 10 (KHÔNG có 11)
    print(x, bin(x), "-> số bit 1 =", bin(x).count("1"))

6 0b110 -> số bit 1 = 2
7 0b111 -> số bit 1 = 3
8 0b1000 -> số bit 1 = 1
9 0b1001 -> số bit 1 = 2
10 0b1010 -> số bit 1 = 2


## 2. Nền tảng Python: hệ nhị phân, `bin()` và `str.count()`

Hệ nhị phân chỉ có 2 chữ số `0, 1`; mỗi vị trí mạnh gấp 2 lần vị trí bên phải.

`10 = 1010₂` vì `1·8 + 0·4 + 1·2 + 0·1 = 10`.

Trong Python:
- `bin(x)` trả về **chuỗi** dạng `'0b1010'`. Tiền tố `'0b'` chỉ để báo "đây là nhị phân".
- `s.count("1")` đếm số lần ký tự `"1"` xuất hiện trong chuỗi `s`. Tiền tố `'0b'` không chứa `'1'` nên không ảnh hưởng.

In [2]:
x = 10
s = bin(x)
print("bin(10)       =", s)
print("kiểu dữ liệu  =", type(s))
print("bỏ tiền tố    =", s[2:])          # slicing: lấy từ vị trí 2 tới hết
print("số bit 1      =", s.count("1"))
print("chuỗi 'banana'.count('a') =", "banana".count("a"))

bin(10)       = 0b1010
kiểu dữ liệu  = <class 'str'>
bỏ tiền tố    = 1010
số bit 1      = 2
chuỗi 'banana'.count('a') = 3


### 2.1. `range(left, right + 1)` — duyệt đoạn đóng

`range(a, b)` sinh các số `a, a+1, ..., b-1` — **không** gồm `b`. Đề hỏi đoạn đóng `[left, right]` nên phải viết `range(left, right + 1)`. Đây là lỗi lệch-một (off-by-one) rất hay gặp.

In [3]:
left, right = 6, 10
print(list(range(left, right)))       # thiếu 10!
print(list(range(left, right + 1)))   # đúng: có cả 10

[6, 7, 8, 9]
[6, 7, 8, 9, 10]


## 3. Nền tảng: số nguyên tố và kiểm tra bằng chia thử

Số nguyên tố là số nguyên `p >= 2` chỉ chia hết cho `1` và chính nó: `2, 3, 5, 7, 11, 13, 17, 19, ...`

- `0` và `1` **không** phải số nguyên tố (bẫy thường gặp với `8 = 1000₂` có 1 bit).
- Để kiểm tra `p`, chỉ cần thử chia cho `d = 2, 3, ...` với `d * d <= p`. Vì nếu `p = a·b` với `a <= b` thì `a <= √p`.

Toán tử `%` cho **phần dư**: `9 % 3 == 0` nghĩa là 9 chia hết cho 3.

In [4]:
def is_prime(p: int) -> bool:
    if p < 2:                 # 0 và 1 không phải số nguyên tố
        return False
    d = 2
    while d * d <= p:         # chỉ thử tới căn bậc hai
        if p % d == 0:        # chia hết -> có ước khác 1 và p
            return False
        d += 1
    return True

print([p for p in range(0, 25) if is_prime(p)])
print("9 % 3 =", 9 % 3, "| 7 % 3 =", 7 % 3)

[2, 3, 5, 7, 11, 13, 17, 19, 23]
9 % 3 = 0 | 7 % 3 = 1


### 3.1. List comprehension `[... for ... if ...]`

Cell trên dùng **list comprehension**: `[p for p in range(0, 25) if is_prime(p)]` đọc là "lấy `p` cho mỗi `p` trong 0..24 **nếu** `is_prime(p)`". Nó tương đương vòng `for` + `append`:

In [5]:
primes = []
for p in range(0, 25):
    if is_prime(p):
        primes.append(p)
print(primes)

[2, 3, 5, 7, 11, 13, 17, 19, 23]


## 4. Nền tảng: toán tử bit `&`, `>>`, mẹo `x & (x - 1)` và `bit_count()`

| Toán tử | Ý nghĩa | Ví dụ |
|---|---|---|
| `a & b` | AND từng bit: bit kết quả = 1 khi **cả hai** bit = 1 | `6 & 3 = 110 & 011 = 010 = 2` |
| `x >> k` | dịch phải `k` bit (bỏ `k` bit thấp nhất) = `x // 2**k` | `10 >> 1 = 5` |
| `x & 1` | lấy bit thấp nhất (0 hoặc 1) | `10 & 1 = 0`, `7 & 1 = 1` |
| `x & (x - 1)` | **xóa bit 1 thấp nhất** của `x` | `12 & 11 = 1100 & 1011 = 1000 = 8` |

Từ Python 3.10 có `x.bit_count()` đếm bit 1 trực tiếp (nhanh, viết bằng C).

In [6]:
print("6 & 3   =", 6 & 3)
print("10 >> 1 =", 10 >> 1, "| 10 & 1 =", 10 & 1, "| 7 & 1 =", 7 & 1)
x = 12
print(f"x       = {x:>2} = {x:04b}")       # :04b in nhị phân, đủ 4 chữ số
print(f"x - 1   = {x-1:>2} = {x-1:04b}")
print(f"x&(x-1) = {x & (x-1):>2} = {x & (x-1):04b}  <- bit 1 thấp nhất đã bị xóa")
print("(21).bit_count() =", (21).bit_count())

6 & 3   = 2
10 >> 1 = 5 | 10 & 1 = 0 | 7 & 1 = 1
x       = 12 = 1100
x - 1   = 11 = 1011
x&(x-1) =  8 = 1000  <- bit 1 thấp nhất đã bị xóa
(21).bit_count() = 3


### 4.1. Vì sao `x & (x - 1)` xóa đúng một bit 1?

Phép `x - 1` "mượn" từ bit 1 thấp nhất: bit đó thành 0, mọi bit 0 phía phải nó thành 1, phần bên trái giữ nguyên. AND với `x` thì phần phải triệt tiêu, phần trái giữ nguyên → mất đúng một bit 1.

Vì vậy đếm số lần làm `x &= x - 1` tới khi `x == 0` chính là số bit 1 (thuật toán **Brian Kernighan**): số vòng lặp = số bit 1, không phải tổng số bit.

In [7]:
def popcount_kernighan(x: int) -> int:
    count = 0
    while x:              # x khác 0 thì còn bit 1
        x &= x - 1        # xóa bit 1 thấp nhất
        count += 1
    return count

for x in [0, 1, 6, 7, 8, 21, 15]:
    print(x, format(x, "b"), popcount_kernighan(x), bin(x).count("1"))

0 0 0 0
1 1 1 1
6 110 2 2
7 111 3 3
8 1000 1 1
21 10101 3 3
15 1111 4 4


## 5. Cách 1 — Trực giác: đếm bit rồi kiểm tra nguyên tố mỗi lần

Ý tưởng thẳng như đề nói:
1. Duyệt mọi `x` từ `left` tới `right`.
2. Đếm số bit 1: `bits = bin(x).count("1")`.
3. Nếu `is_prime(bits)` thì tăng biến đếm.

In [8]:
def count_prime_set_bits_naive(left: int, right: int) -> int:
    count = 0
    for x in range(left, right + 1):
        bits = bin(x).count("1")
        if is_prime(bits):
            count += 1
    return count

print(count_prime_set_bits_naive(6, 10))    # kỳ vọng 4
print(count_prime_set_bits_naive(10, 15))   # kỳ vọng 5

4
5


### Code nộp LeetCode — Cách 1

In [9]:
class Solution:
    def countPrimeSetBits(self, left: int, right: int) -> int:
        def is_prime(p: int) -> bool:
            if p < 2:
                return False
            d = 2
            while d * d <= p:
                if p % d == 0:
                    return False
                d += 1
            return True

        count = 0
        for x in range(left, right + 1):
            if is_prime(bin(x).count("1")):
                count += 1
        return count

print(Solution().countPrimeSetBits(6, 10), Solution().countPrimeSetBits(10, 15))

4 5


**Độ phức tạp cách 1:** gọi `n = right - left + 1` (≤ 10⁴ + 1), `B` = số bit của `right` (≤ 20).
- Đếm bit: `O(B)`; kiểm tra nguyên tố của `bits ≤ B`: `O(√B)`.
- Tổng: `O(n · B)` thời gian, `O(B)` bộ nhớ phụ (chuỗi `bin`).

Đã đủ nhanh, nhưng ta đang **lặp lại** kiểm tra nguyên tố cho cùng vài giá trị nhỏ hàng nghìn lần.

## 6. Nút thắt và insight — số bit 1 luôn ≤ 20

Bao nhiêu bit cần để viết `10^6`? Vì `2^20 = 1 048 576 > 10^6`, mọi `x <= 10^6` có **tối đa 20 bit**, nên số bit 1 nằm trong `0..20`.

→ Các số nguyên tố có thể xuất hiện chỉ là `{2, 3, 5, 7, 11, 13, 17, 19}`. Ta **tính sẵn một lần** (hoặc viết cứng) tập này, rồi tra `bits in PRIMES` trong `O(1)`.

In [10]:
print("2**20 =", 2**20, "> 10**6 ?", 2**20 > 10**6)
print("số bit của 10**6 =", (10**6).bit_length())
print("bit 1 nhiều nhất trong [1, 10**6]:", max(x.bit_count() for x in range(1, 10**6 + 1)))
print("2**20 - 1 =", 2**20 - 1, "có", (2**20 - 1).bit_count(), "bit 1 nhưng > 10**6 nên không xuất hiện")
PRIMES = {p for p in range(21) if is_prime(p)}
print("PRIMES =", sorted(PRIMES))

2**20 = 1048576 > 10**6 ? True
số bit của 10**6 = 20
bit 1 nhiều nhất trong [1, 10**6]: 19
2**20 - 1 = 1048575 có 20 bit 1 nhưng > 10**6 nên không xuất hiện
PRIMES = [2, 3, 5, 7, 11, 13, 17, 19]


### 6.1. `set` — tập hợp, tra cứu `in` nhanh

`{2, 3, 5}` là một **set**: tập không trùng lặp, không có thứ tự. Phép `x in s` trung bình `O(1)` (băm/hash), trong khi `x in list` phải duyệt `O(len)`.

`{p for p in ...}` là **set comprehension**, giống list comprehension nhưng tạo set.

In [11]:
S = {2, 3, 5, 7, 11, 13, 17, 19}
print(3 in S, 4 in S, 19 in S, 1 in S)
print(type(S), len(S))

True False True False
<class 'set'> 8


## 7. Lời giải chính — tập nguyên tố cố định + đếm bit

Ghép lại:
```python
PRIMES = {2, 3, 5, 7, 11, 13, 17, 19}
count = 0
for x in range(left, right + 1):   # (1) duyệt đoạn đóng
    if x.bit_count() in PRIMES:    # (2) đếm bit, (3) tra tập
        count += 1                 # (4) đạt thì đếm
return count
```

- `(1)` nhớ `+ 1` để gồm `right`.
- `(2)` `bit_count()` (Python ≥ 3.10). Nếu môi trường cũ, dùng `bin(x).count("1")`.
- `(3)` tra set `O(1)`.

In [12]:
PRIMES_UPTO_20 = {2, 3, 5, 7, 11, 13, 17, 19}

def count_prime_set_bits(left: int, right: int) -> int:
    count = 0
    for x in range(left, right + 1):
        if x.bit_count() in PRIMES_UPTO_20:
            count += 1
    return count

print(count_prime_set_bits(6, 10), count_prime_set_bits(10, 15))

4 5


### 7.1. Viết gọn bằng `sum(...)` với generator

`True` trong Python cư xử như số `1`, `False` như `0`. Vì vậy `sum(điều_kiện for x in ...)` đếm số lần điều kiện đúng.

In [13]:
print(True + True + False)
def count_prime_set_bits_sum(left: int, right: int) -> int:
    return sum(x.bit_count() in PRIMES_UPTO_20 for x in range(left, right + 1))
print(count_prime_set_bits_sum(6, 10), count_prime_set_bits_sum(10, 15))

2
4 5


## 8. TRACE từng bước

Bảng trace ví dụ 2 (`left = 10, right = 15`):

| x | nhị phân | bits | bits in PRIMES? | count sau bước |
|---:|---|:---:|:---:|:---:|
| 10 | 1010 | 2 | có | 1 |
| 11 | 1011 | 3 | có | 2 |
| 12 | 1100 | 2 | có | 3 |
| 13 | 1101 | 3 | có | 4 |
| 14 | 1110 | 3 | có | 5 |
| 15 | 1111 | 4 | không | 5 |

Hàm trace in đúng các cột này, kể cả giá trị `count` **trước** và **sau** khi quyết định.

In [14]:
def trace_count_prime_set_bits(left: int, right: int) -> int:
    count = 0
    print(" x  | nhị phân | bits | nguyên tố? | count trước -> sau")
    for x in range(left, right + 1):
        bits = x.bit_count()
        ok = bits in PRIMES_UPTO_20
        before = count
        if ok:
            count += 1
        print(f"{x:>3} | {x:>8b} | {bits:>4} | {str(ok):>10} | {before} -> {count}")
    print("Kết quả:", count)
    return count

trace_count_prime_set_bits(10, 15)
print()
trace_count_prime_set_bits(6, 10)

 x  | nhị phân | bits | nguyên tố? | count trước -> sau
 10 |     1010 |    2 |       True | 0 -> 1
 11 |     1011 |    3 |       True | 1 -> 2
 12 |     1100 |    2 |       True | 2 -> 3
 13 |     1101 |    3 |       True | 3 -> 4
 14 |     1110 |    3 |       True | 4 -> 5
 15 |     1111 |    4 |      False | 5 -> 5
Kết quả: 5

 x  | nhị phân | bits | nguyên tố? | count trước -> sau
  6 |      110 |    2 |       True | 0 -> 1
  7 |      111 |    3 |       True | 1 -> 2
  8 |     1000 |    1 |      False | 2 -> 2
  9 |     1001 |    2 |       True | 2 -> 3
 10 |     1010 |    2 |       True | 3 -> 4
Kết quả: 4


4

## 9. Cách 3 — Bitmask `665772`: tập nguyên tố nằm trong một số nguyên

Thay vì set, ta "mã hóa" tập `{2,3,5,7,11,13,17,19}` vào **một số**: bật bit thứ `p` cho mỗi số nguyên tố `p`.

`MASK = 2² + 2³ + 2⁵ + 2⁷ + 2¹¹ + 2¹³ + 2¹⁷ + 2¹⁹ = 665772`.

Muốn hỏi "`k` có nguyên tố không?" → xem bit thứ `k` của `MASK`: `(MASK >> k) & 1`.

`1 << p` là dịch trái = `2**p`; `|` là OR từng bit (gộp bit).

In [15]:
MASK = 0
for p in [2, 3, 5, 7, 11, 13, 17, 19]:
    MASK |= 1 << p          # bật bit p
print("MASK =", MASK)
print("MASK nhị phân (bit 0 ở bên phải):", format(MASK, "b"))
for k in range(0, 21):
    print(k, (MASK >> k) & 1, end=" | ")

MASK = 665772
MASK nhị phân (bit 0 ở bên phải): 10100010100010101100
0 0 | 1 0 | 2 1 | 3 1 | 4 0 | 5 1 | 6 0 | 7 1 | 8 0 | 9 0 | 10 0 | 11 1 | 12 0 | 13 1 | 14 0 | 15 0 | 16 0 | 17 1 | 18 0 | 19 1 | 20 0 | 

In [16]:
class Solution:
    def countPrimeSetBits(self, left: int, right: int) -> int:
        MASK = 665772            # bit k bật  <=>  k là số nguyên tố (k <= 20)
        return sum((MASK >> x.bit_count()) & 1 for x in range(left, right + 1))

print(Solution().countPrimeSetBits(6, 10), Solution().countPrimeSetBits(10, 15))

4 5


## 10. Cách 4 — DP đếm bit: `bits[x] = bits[x >> 1] + (x & 1)`

Nếu không có `bit_count()`, ta có thể tính số bit 1 cho **mọi** số `0..right` bằng quy hoạch động (dynamic programming):

- `x >> 1` là `x` bỏ bit cuối → đã tính trước đó (nhỏ hơn `x`).
- `x & 1` là bit cuối.

Ví dụ `bits[13] = bits[6] + 1` vì `13 = 1101`, `6 = 110`.

Cách này tốn `O(right)` bộ nhớ (tới 10⁶ phần tử) — đắt hơn khi đoạn `[left, right]` ngắn, nhưng hữu ích khi cần trả lời **rất nhiều truy vấn**.

In [17]:
def count_prime_set_bits_dp(left: int, right: int) -> int:
    bits = [0] * (right + 1)                 # list right+1 số 0
    for x in range(1, right + 1):
        bits[x] = bits[x >> 1] + (x & 1)
    return sum(bits[x] in PRIMES_UPTO_20 for x in range(left, right + 1))

demo = [0] * 14
for x in range(1, 14):
    demo[x] = demo[x >> 1] + (x & 1)
print("bits[0..13] =", demo)
print(count_prime_set_bits_dp(6, 10), count_prime_set_bits_dp(10, 15))

bits[0..13] = [0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3]
4 5


### Code nộp LeetCode — lời giải chính (khuyến nghị)

In [18]:
class Solution:
    def countPrimeSetBits(self, left: int, right: int) -> int:
        primes = {2, 3, 5, 7, 11, 13, 17, 19}
        count = 0
        for x in range(left, right + 1):
            if bin(x).count("1") in primes:
                count += 1
        return count

sol = Solution()
assert sol.countPrimeSetBits(6, 10) == 4
assert sol.countPrimeSetBits(10, 15) == 5
print("Solution OK")

Solution OK


## 11. Vì sao đúng?

**Invariant (bất biến) của vòng lặp:** sau khi xử lý xong các số `left..x`, biến `count` bằng đúng số lượng số trong `[left, x]` có số bit 1 là nguyên tố.

- Ban đầu (chưa xử lý số nào): `count = 0` — đúng.
- Bước `x`: ta cộng 1 **khi và chỉ khi** `bits(x)` nguyên tố → invariant giữ cho `[left, x]`.
- Kết thúc ở `x = right` → `count` là đáp án.

**Vì sao tập `{2,...,19}` đủ?** `x <= 10^6 < 2^20` nên `bits(x) <= 20`; mọi số nguyên tố `<= 20` đều có trong tập. Không có số nguyên tố nào bị thiếu, và không có số không nguyên tố nào trong tập → kiểm tra `in` cho kết quả giống `is_prime`.

**Bitmask:** bit thứ `k` của `665772` bằng 1 đúng khi `k` thuộc tập trên → `(MASK >> k) & 1` tương đương `k in PRIMES` với `k <= 20`.

In [19]:
# Kiểm chứng: với mọi k từ 0..20, ba cách kiểm tra cho cùng kết quả
for k in range(21):
    assert is_prime(k) == (k in PRIMES_UPTO_20) == bool((665772 >> k) & 1)
print("is_prime, set, bitmask khớp nhau cho k = 0..20")

is_prime, set, bitmask khớp nhau cho k = 0..20


## 12. Độ phức tạp và so sánh các cách

Gọi `n = right - left + 1` (≤ 10⁴ + 1), `B = 20` (số bit tối đa).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Trực giác | `bin().count` + `is_prime` mỗi số | `O(n·B)` | `O(B)` | Dễ nghĩ nhất |
| 2. Tập nguyên tố | `bit_count()` + `in` set | `O(n)` với `bit_count` (coi như O(1) cho số ≤ 20 bit) | `O(1)` | **Khuyến nghị** |
| 3. Bitmask 665772 | `(MASK >> bits) & 1` | `O(n)` | `O(1)` | Gọn, "mẹo" |
| 4. DP đếm bit | `bits[x] = bits[x>>1] + (x&1)` | `O(right)` | `O(right)` | Hợp khi nhiều truy vấn |

Với `bin(x).count("1")` thì mỗi số tốn `O(B)` → `O(n·B)`; vì `B ≤ 20` là hằng số nhỏ, thực tế rất nhanh.

In [20]:
import time
for f in [count_prime_set_bits_naive, count_prime_set_bits, count_prime_set_bits_sum, count_prime_set_bits_dp]:
    t = time.perf_counter()
    r = f(990000, 1000000)
    print(f"{f.__name__:<30} -> {r}   ({(time.perf_counter() - t) * 1000:.1f} ms)")

count_prime_set_bits_naive     -> 3754   (2.7 ms)
count_prime_set_bits           -> 3754   (0.5 ms)
count_prime_set_bits_sum       -> 3754   (0.5 ms)


count_prime_set_bits_dp        -> 3754   (61.7 ms)


## 13. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu nhiều cách

Nhóm test:
1. Ví dụ đề: `(6,10) → 4`, `(10,15) → 5`.
2. Biên nhỏ: `(1,1)`: 1 = `1` có 1 bit → 0. `(3,3)`: `11` có 2 bit → 1.
3. Phản ví dụ cho lỗi "coi 1 là nguyên tố": `(8,8)` → 0.
4. Phản ví dụ cho lỗi `range(left, right)`: `(7,7)` → 1 (nếu quên `+1` sẽ ra 0).
5. Biên lớn: `right = 10^6`.
6. Đối chiếu 4 cách trên nhiều đoạn ngẫu nhiên.

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", count_prime_set_bits(6, 10), 4)
check("đề 2", count_prime_set_bits(10, 15), 5)
check("x=1 (1 bit)", count_prime_set_bits(1, 1), 0)
check("x=3 (2 bit)", count_prime_set_bits(3, 3), 1)
check("x=8: 1 không nguyên tố", count_prime_set_bits(8, 8), 0)
check("x=7: đoạn một phần tử", count_prime_set_bits(7, 7), 1)
check("biên lớn 10^6", count_prime_set_bits(10**6, 10**6), count_prime_set_bits_naive(10**6, 10**6))

PASS đề 1: 4
PASS đề 2: 5
PASS x=1 (1 bit): 0
PASS x=3 (2 bit): 1
PASS x=8: 1 không nguyên tố: 0
PASS x=7: đoạn một phần tử: 1
PASS biên lớn 10^6: 1


In [22]:
import random
random.seed(762)
for _ in range(200):
    left = random.randint(1, 10**6)
    right = min(10**6, left + random.randint(0, 300))
    expected = count_prime_set_bits_naive(left, right)
    assert count_prime_set_bits(left, right) == expected
    assert count_prime_set_bits_sum(left, right) == expected
    assert Solution().countPrimeSetBits(left, right) == expected
print("200 đoạn ngẫu nhiên: các cách khớp nhau")
assert count_prime_set_bits_dp(1, 5000) == count_prime_set_bits_naive(1, 5000)
print("DP khớp trên [1, 5000]")

200 đoạn ngẫu nhiên: các cách khớp nhau
DP khớp trên [1, 5000]


## 14. Lỗi người mới hay gặp

### Lỗi 1: Quên `+ 1` trong `range`
```python
for x in range(left, right):   # SAI: bỏ mất right
```
→ Với `left = right = 7` trả 0 thay vì 1. Sửa: `range(left, right + 1)`.

### Lỗi 2: Coi 1 là số nguyên tố
```python
primes = {1, 2, 3, 5, 7, ...}   # SAI
```
→ `8 = 1000` sẽ bị đếm nhầm. Số nguyên tố bắt đầu từ 2.

### Lỗi 3: Kiểm tra nguyên tố cho `x` thay vì cho số bit
```python
if is_prime(x): ...   # SAI: đề hỏi số bit 1 của x
```

### Lỗi 4: Tập nguyên tố thiếu (chỉ tới 13 hay 17)
Số `2^19 - 1 = 524287 <= 10^6` có **19** bit 1 → phải có 19 trong tập.

In [23]:
def wrong_range(left, right):
    return sum(x.bit_count() in PRIMES_UPTO_20 for x in range(left, right))

def wrong_one_is_prime(left, right):
    return sum(x.bit_count() in {1, 2, 3, 5, 7, 11, 13, 17, 19} for x in range(left, right + 1))

def wrong_short_set(left, right):
    return sum(x.bit_count() in {2, 3, 5, 7, 11, 13, 17} for x in range(left, right + 1))

print("(7,7):    sai =", wrong_range(7, 7), "| đúng =", count_prime_set_bits(7, 7))
print("(8,8):    sai =", wrong_one_is_prime(8, 8), "| đúng =", count_prime_set_bits(8, 8))
x = 2**19 - 1
print(f"{x} có {x.bit_count()} bit: sai =", wrong_short_set(x, x), "| đúng =", count_prime_set_bits(x, x))

(7,7):    sai = 0 | đúng = 1
(8,8):    sai = 1 | đúng = 0
524287 có 19 bit: sai = 0 | đúng = 1


## 15. Mẹo nhận dạng pattern

- Đề nói "**set bits**", "số bit 1", "Hamming weight" → nghĩ tới `bit_count()`, `bin().count('1')`, `x & (x-1)`.
- Một thuộc tính được hỏi lặp lại cho **giá trị nằm trong miền nhỏ** (ở đây 0..20) → **tính sẵn** (precompute) một tập/bảng/bitmask.
- Constraints `right - left <= 10^4` → duyệt từng số là đủ, không cần công thức tổ hợp.
- "Tập nhỏ các số `< 64`" → có thể nén vào **bitmask** một số nguyên.

### Biến thể

1. **Đếm số có số bit 1 là số chẵn** (thay tập nguyên tố bằng điều kiện `% 2 == 0`).
2. **Liệt kê** thay vì đếm.
3. **Nhiều truy vấn** `(left, right)`: dùng mảng cộng dồn (prefix sum) `pre[x]` = số lượng số tốt trong `[1, x]`; trả lời `pre[right] - pre[left-1]` trong `O(1)`.
4. **Counting Bits (LeetCode 338)**: trả về số bit 1 của mọi số `0..n` — chính là cách 4.

In [24]:
# Biến thể 3: prefix sum cho nhiều truy vấn
N = 2000
good = [0] * (N + 1)
for x in range(1, N + 1):
    good[x] = 1 if x.bit_count() in PRIMES_UPTO_20 else 0
pre = [0] * (N + 1)
for x in range(1, N + 1):
    pre[x] = pre[x - 1] + good[x]      # pre[x] = số tốt trong [1, x]

for (l, r) in [(6, 10), (10, 15), (100, 1999)]:
    print((l, r), "->", pre[r] - pre[l - 1], "| kiểm tra:", count_prime_set_bits(l, r))

(6, 10) -> 4 | kiểm tra: 4
(10, 15) -> 5 | kiểm tra: 5
(100, 1999) -> 939 | kiểm tra: 939


## 16. Bài tập tự luyện (TODO — tự làm trước, đáp án ở cuối)

**Bài 1 (tính tay):** Không chạy code, liệt kê nhị phân và số bit 1 của các số `16..20`, rồi cho biết `countPrimeSetBits(16, 20)` bằng bao nhiêu. Ghi đáp án vào biến `answer_ex1`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `popcount_exercise(x)` dùng vòng `while` và `x & 1`, `x >>= 1` (không dùng `bin` hay `bit_count`).

**Bài 3 (viết lại):** Viết `count_prime_set_bits_exercise(left, right)` dùng bitmask `665772`.

**Bài 4 (biến thể):** Viết `count_even_set_bits(left, right)`: đếm số trong đoạn có số bit 1 là **số chẵn** (0 được tính là chẵn).

**Bài 5 (mini challenge):** Viết `list_prime_set_bits(left, right)` trả về **list các số** thỏa đề, theo thứ tự tăng.

In [25]:
answer_ex1 = None   # TODO: điền số nguyên

def popcount_exercise(x: int) -> int:
    # TODO 1: count = 0
    # TODO 2: while x > 0: cộng bit thấp nhất (x & 1) vào count, rồi dịch phải x
    pass

# assert popcount_exercise(21) == 3

In [26]:
def count_prime_set_bits_exercise(left: int, right: int) -> int:
    # TODO: MASK = 665772; dùng (MASK >> bits) & 1
    pass

def count_even_set_bits(left: int, right: int) -> int:
    # TODO
    pass

def list_prime_set_bits(left: int, right: int) -> list[int]:
    # TODO
    pass

# assert count_prime_set_bits_exercise(6, 10) == 4
# assert count_even_set_bits(1, 3) == 1
# assert list_prime_set_bits(6, 10) == [6, 7, 9, 10]

### Gợi ý 3 mức

**Bài 1** — (1) Viết từng số ra nhị phân 5 chữ số. (2) `16 = 10000`. (3) Chỉ số nào có 2 hoặc 3 bit 1 mới đạt.

**Bài 2** — (1) Mỗi vòng xem bit cuối rồi bỏ nó. (2) `x & 1` cho bit cuối, `x >>= 1` bỏ bit cuối. (3) `while x: count += x & 1; x >>= 1`.

**Bài 3** — (1) Thay `in PRIMES` bằng tra bit. (2) Dịch phải `MASK` đúng `bits` lần. (3) `sum((665772 >> x.bit_count()) & 1 for x in range(left, right + 1))`.

**Bài 4** — (1) Chẵn ⇔ chia 2 dư 0. (2) `x.bit_count() % 2 == 0`. (3) Dùng `sum(...)` với generator.

**Bài 5** — (1) Giống đếm nhưng `append` thay vì `+= 1`. (2) List comprehension có `if`. (3) `[x for x in range(...) if ...]`.

---

## 17. Đáp án tham khảo

Chỉ xem sau khi đã tự làm.

In [27]:
# Bài 1: 16=10000(1), 17=10001(2), 18=10010(2), 19=10011(3), 20=10100(2) -> 4 số đạt
answer_ex1_solution = 4

def popcount_solution(x: int) -> int:
    count = 0
    while x > 0:
        count += x & 1
        x >>= 1
    return count

def count_prime_set_bits_solution(left: int, right: int) -> int:
    MASK = 665772
    return sum((MASK >> x.bit_count()) & 1 for x in range(left, right + 1))

def count_even_set_bits_solution(left: int, right: int) -> int:
    return sum(x.bit_count() % 2 == 0 for x in range(left, right + 1))

def list_prime_set_bits_solution(left: int, right: int) -> list[int]:
    return [x for x in range(left, right + 1) if x.bit_count() in PRIMES_UPTO_20]

for x in range(16, 21):
    print(x, format(x, "05b"), x.bit_count())
assert answer_ex1_solution == count_prime_set_bits(16, 20)
assert all(popcount_solution(x) == x.bit_count() for x in range(2000))
assert count_prime_set_bits_solution(6, 10) == 4 and count_prime_set_bits_solution(10, 15) == 5
assert count_even_set_bits_solution(1, 3) == 1          # chỉ 3 = 11 có 2 bit
assert list_prime_set_bits_solution(6, 10) == [6, 7, 9, 10]
print("Đáp án PASS toàn bộ test!")

16 10000 1
17 10001 2
18 10010 2
19 10011 3
20 10100 2
Đáp án PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def countPrimeSetBits(self, left: int, right: int) -> int:
        primes = {2, 3, 5, 7, 11, 13, 17, 19}     # đủ vì right <= 10^6 < 2^20
        return sum(bin(x).count("1") in primes for x in range(left, right + 1))
```

Công thức cần nhớ:
- Đếm bit 1: `bin(x).count("1")` · `x.bit_count()` · `while x: x &= x - 1`.
- Bit thứ `k`: `(x >> k) & 1`. Bật bit: `mask |= 1 << k`.
- DP đếm bit: `bits[x] = bits[x >> 1] + (x & 1)`.

**Checklist trước khi submit:**
- [ ] Dùng `range(left, right + 1)` (đoạn đóng)?
- [ ] Tập nguyên tố **không** chứa 0, 1 và có đủ tới 19?
- [ ] Kiểm tra nguyên tố cho **số bit**, không phải cho `x`?
- [ ] Đã thử `(6,10) → 4`, `(10,15) → 5`, `(8,8) → 0`, `(7,7) → 1`?
- [ ] Nếu nền tảng cũ (< Python 3.10) thì dùng `bin(x).count("1")` thay `bit_count()`.